# CIFAR-10 Long-Tail (LT) 불균형 분류 실험

---

## 1. 태스크 및 도메인
- **도메인**: CIFAR-10 Long-Tail (인공 불균형) 이미지 분류
- **모달리티**: RGB 컬러 이미지 (32×32)
- **태스크**: 10-class 분류
  - 클래스: 비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭
- **핵심 도전**: 극단적 클래스 불균형 (IR=10/50/100) 하에서 손실 함수 효과 검증

## 2. 모델
- **아키텍처**: ResNet-32 (He et al. 2016 CIFAR 전용)
- **사전학습**: 없음 (CIFAR-LT는 scratch 학습이 표준)
- **선택 이유**: CIFAR-LT 벤치마크 표준 모델 (LDAM, BBN, cRT 등 논문 기준선)
- **출력**: 10채널 softmax logits

## 3. 데이터셋
- **이름**: CIFAR-10 Long-Tail (torchvision + 지수 감소 서브샘플링)
- **규모**: 
  - Original CIFAR-10: 50,000 train / 10,000 test (클래스당 5,000 / 1,000)
  - LT 변환: IR=100 → train 클래스당 5,000~50장 (불균형), test는 균형 유지
- **입력 해상도**: 32×32 RGB
- **클래스 불균형**: 
  - IR=10: max 5,000 / min 500 (비교적 완만)
  - IR=50: max 5,000 / min 100
  - IR=100: max 5,000 / min 50 (극심)
- **공식 분할**: 없음 → 8:1:1 (train 40,000 / val 5,000 / test 10,000)

## 4. 데이터 준비 (협업자용)
> Cell 0 자동 실행 시 torchvision으로 자동 다운로드됩니다.

**취득 방법**:
- `torchvision.datasets.CIFAR10(download=True)` — Cell 0 실행 시 자동 다운로드

**Colab 환경**: 설치 불필요 (torchvision 사전 설치됨)

## 5. 전처리 및 데이터 특이점
- **Augmentation (학습)**: RandomCrop(32, padding=4) + RandomHorizontalFlip
- **정규화**: ImageNet 기준 mean/std 사용 (CIFAR-LT 논문 표준)
  - mean=[0.4914, 0.4822, 0.4465]
  - std=[0.2023, 0.1994, 0.2010]
- **불균형 생성**: 지수 감소 분포 — n_i = n_max × IR^(-i/(K-1))
  - n_max = 5,000 (원본 클래스당 샘플 수)
  - K = 10 (클래스 수)
- **Test Set**: 원본 균형 유지 (각 클래스 1,000장) — 공정한 평가

## 6. 실험 손실 함수 및 Optuna 탐색 범위
| 손실 함수 | 탐색 파라미터 | 탐색 범위 | Trials |
|-----------|-------------|----------|--------|
| `ce` | — | — | — |
| `wce` | — | — | — |
| `lwce` | — | — | — |
| `plwce` | alpha | 2.5 ~ 15.0 | 20 (1D GridSampler) |
| `cb` | — | — | — |
| `plwce_focal` | alpha + gamma | alpha 2.5~15.0(8) × gamma 1.0~5.0(5) | 40 (2D GridSampler) |

**Optuna 설정** (proxy learning):
- subset_ratio=0.20 (전체 LT 학습셋의 20%만 사용)
- proxy_epochs=40 (빠른 탐색)
- metric: Balanced Accuracy (macro recall, 불균형 환경에서 핵심)

## 7. SoTA 참고 (2025년 12월 기준)
| 방법 | Top-1 Acc (%) | Balanced Acc (%) | Few-shot Acc (%) | 출처 |
|------|-------------|-----------------|------------------|------|
| LDAM (2019) | 72.58 (IR=100) | — | — | ICML'19 |
| BBN (2019) | 73.41 (IR=100) | — | — | ICCV'19 |
| cRT (2020) | 75.19 (IR=100) | — | — | ICML'21 |
| Decoupling (2019) | 76.46 (IR=100) | — | — | ICCV'19 |
| 임의 U-Net (CIFAR-LT 비표준) | ~65 (IR=100) | — | — | baseline |

> 본 연구 목표: ResNet-32 + 표준 SGD 학습 하에서 LWCE/PLWCE 손실함수의 개선 효과 검증.
> 평가 지표: Top-1 정확도 + Balanced Accuracy (macro recall) + Few-shot Accuracy
> 결과 저장: `image_classification/results/CIFAR10_LT/IR{ir}/`


In [2]:
# === Cell 0: 환경 설정 ===

!pip install optuna torchvision pandas openpyxl -q

import os, sys, json, pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from sklearn.metrics import confusion_matrix, f1_score

import optuna
from optuna.samplers import GridSampler

from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/drive/MyDrive/imbalanced-data-LWCE'
# custom_losses.py / experiment_utils.py 는 REPO 루트, resnet32.py 는 image_classification/.
sys.path.insert(0, f'{REPO}/image_classification')
sys.path.insert(0, REPO)

from custom_losses import get_clf_loss
from experiment_utils import (GradLogger, extended_metrics, OPTUNA_GRIDS,
                              grid_n_trials, LOSS_CONFIGS_11, ABLATION_ROWS)
from resnet32 import build_resnet32

# --- 상수 설정 ---
DATASET = 'cifar10'
NUM_CLASSES = 10
IR_LIST = [10, 50, 100]

BATCH_SIZE = 128
NUM_WORKERS = 0
SEED = 42
FINAL_EPOCHS = 200
SEEDS = [42, 43, 44, 45, 46]

# 3차 피드백: §4.2 baseline에 logitadj 추가, §4.5 ablation에 combined 추가,
#             eslwce는 논문 proposed 3형제 중 하나라 필수.
LOSS_CONFIGS = LOSS_CONFIGS_11   # 11종: 기존 8 + eslwce/combined/logitadj

RESULTS_BASE = f'{REPO}/image_classification/results/CIFAR10_LT'
os.makedirs(RESULTS_BASE, exist_ok=True)

CKPT_OPTUNA        = f'{RESULTS_BASE}/optuna_checkpoint.json'
CKPT_OPTUNA_TRIALS = f'{RESULTS_BASE}/optuna_trials.json'
CKPT_RESULTS       = f'{RESULTS_BASE}/results_checkpoint.json'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device} | '
      f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
matplotlib.use('Agg')

print(f'✓ Losses ({len(LOSS_CONFIGS)}): {LOSS_CONFIGS}')
print(f'✓ Total runs: {len(IR_LIST)} IRs × {len(LOSS_CONFIGS)} losses × {len(SEEDS)} seeds '
      f'= {len(IR_LIST)*len(LOSS_CONFIGS)*len(SEEDS)} runs')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ Google Drive 마운트 완료
✓ 모듈 로드 성공: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification
✓ Device: cuda
  GPU: Tesla T4
  Memory: 15.6 GB

✓ 환경 설정 완료
  Seeds: [42, 43, 44, 45, 46]
  Total runs: 3 IRs × 6 losses × 5 seeds = 90 runs


In [3]:
# === Cell 1: CIFAR-LT 데이터셋 생성 및 로드 ===

def make_cifar_lt(dataset_name: str, imbalance_ratio: int, seed: int = 42):
    """
    CIFAR 데이터셋을 불균형(long-tail) 분포로 변환.
    
    지수 감소: n_i = n_max × IR^(-i/(K-1))
    
    Args:
        dataset_name: 'cifar10' or 'cifar100'
        imbalance_ratio: IR=10, 50, 100 등
        seed: 재현성
        
    Returns:
        indices (train LT indices), class_counts (list of K elements)
    """
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    
    # 전체 데이터셋 다운로드
    if dataset_name == 'cifar10':
        dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    else:
        dataset = datasets.CIFAR100(root='/tmp/cifar', train=True, download=True, transform=None)
    
    targets = np.array(dataset.targets)
    
    # 클래스별 인덱스 그룹화
    class_indices = [np.where(targets == c)[0] for c in range(K)]
    
    # 불균형 수정: n_i 계산
    rho = imbalance_ratio ** (-1 / (K - 1))
    class_counts = [int(n_max * (rho ** i)) for i in range(K)]
    
    # 각 클래스에서 n_i개씩 랜덤 선택
    np.random.seed(seed)
    lt_indices = []
    for c, n_samples in enumerate(class_counts):
        n_samples = max(1, n_samples)  # 최소 1개
        selected = np.random.choice(class_indices[c], size=n_samples, replace=False)
        lt_indices.extend(selected)
    
    lt_indices = np.array(lt_indices)
    np.random.shuffle(lt_indices)
    
    return lt_indices.tolist(), class_counts


# --- CIFAR-10 train/val/test splits ---
def load_cifar_lt_loaders(ir: int, batch_size: int = 128, num_workers: int = 0):
    """
    CIFAR-10 LT + standard CIFAR-10 test을 로드.
    Train LT set을 80/20으로 나눔 (val은 Optuna/early stopping용, test는 최종 평가용)
    """
    # LT train 생성
    full_dataset = datasets.CIFAR10(root='/tmp/cifar', train=True, download=True, transform=None)
    lt_indices, class_counts = make_cifar_lt('cifar10', ir, seed=SEED)
    lt_indices = np.array(lt_indices)  # Convert to numpy array for proper indexing
    
    # train/val 분할 (stratified)
    lt_targets = np.array(full_dataset.targets)[lt_indices]
    n_val = len(lt_indices) // 5  # 20%
    
    # 클래스별로 stratified split
    train_indices, val_indices = [], []
    for c in range(10):
        c_mask = lt_targets == c
        c_idx = np.where(c_mask)[0]
        np.random.seed(SEED)
        np.random.shuffle(c_idx)
        n_c_val = max(1, len(c_idx) // 5)
        val_indices.extend(lt_indices[c_idx[:n_c_val]])
        train_indices.extend(lt_indices[c_idx[n_c_val:]])
    
    # Transform 설정
    train_tf = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                             std=[0.2023, 0.1994, 0.2010]),
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465],
                             std=[0.2023, 0.1994, 0.2010]),
    ])
    
    # Datasets with transform
    train_ds = Subset(full_dataset, train_indices)
    train_ds.dataset.transform = train_tf
    
    val_ds = Subset(full_dataset, val_indices)
    val_ds.dataset.transform = test_tf
    
    test_ds = datasets.CIFAR10(root='/tmp/cifar', train=False, download=True, transform=test_tf)
    
    # DataLoaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    
    return train_loader, val_loader, test_loader, class_counts


print('✓ CIFAR-LT 함수 정의 완료')

✓ CIFAR-LT 함수 정의 완료


In [4]:
# === Cell 2: 클래스 분포 시각화 ===

def visualize_class_distribution(class_counts, ir, dataset_name='CIFAR-10'):
    """
    불균형 분포 시각화 및 그룹 경계 출력.
    """
    counts_arr = np.array(class_counts)
    
    print(f'\n{"="*60}')
    print(f'{dataset_name} LT (IR={ir}) — 클래스 분포')
    print(f'{"="*60}')
    print(f'Min: {counts_arr.min():5d} | Max: {counts_arr.max():5d} | Ratio: {counts_arr.max()/counts_arr.min():.1f}:1')
    print(f'Total samples: {counts_arr.sum():,}')
    
    # Many/Medium/Few 그룹 계산
    many_mask = counts_arr >= 100
    medium_mask = (counts_arr >= 20) & (counts_arr < 100)
    few_mask = counts_arr < 20
    
    print(f'\nGroup distribution:')
    print(f'  Many-shot (n≥100):   {many_mask.sum():2d} classes')
    print(f'  Medium-shot (20≤n):  {medium_mask.sum():2d} classes')
    print(f'  Few-shot (n<20):     {few_mask.sum():2d} classes')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(12, 4))
    colors = ['green' if m else ('orange' if med else 'red') 
              for m, med in zip(many_mask, medium_mask)]
    ax.bar(range(len(class_counts)), class_counts, color=colors, alpha=0.7)
    ax.set_xlabel('Class')
    ax.set_ylabel('# Samples (log scale)', fontsize=11)
    ax.set_yscale('log')
    ax.set_title(f'{dataset_name} LT Distribution (IR={ir})', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    
    ir_dir = f'{RESULTS_BASE}/IR{ir}'
    os.makedirs(ir_dir, exist_ok=True)
    plt.savefig(f'{ir_dir}/class_distribution.png', dpi=100, bbox_inches='tight')
    plt.close()
    print(f'\n✓ 분포 시각화 저장: {ir_dir}/class_distribution.png')


# Test with first IR
for ir in IR_LIST:
    os.makedirs(f'{RESULTS_BASE}/IR{ir}', exist_ok=True)
    _, _, _, class_counts = load_cifar_lt_loaders(ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
    visualize_class_distribution(class_counts, ir)

100%|██████████| 170M/170M [00:13<00:00, 12.5MB/s] 



CIFAR-10 LT (IR=10) — 클래스 분포
Min:   500 | Max:  5000 | Ratio: 10.0:1
Total samples: 20,431

Group distribution:
  Many-shot (n≥100):   10 classes
  Medium-shot (20≤n):   0 classes
  Few-shot (n<20):      0 classes

✓ 분포 시각화 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT/IR10/class_distribution.png

CIFAR-10 LT (IR=50) — 클래스 분포
Min:    99 | Max:  5000 | Ratio: 50.5:1
Total samples: 13,995

Group distribution:
  Many-shot (n≥100):    9 classes
  Medium-shot (20≤n):   1 classes
  Few-shot (n<20):      0 classes

✓ 분포 시각화 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT/IR50/class_distribution.png

CIFAR-10 LT (IR=100) — 클래스 분포
Min:    50 | Max:  5000 | Ratio: 100.0:1
Total samples: 12,406

Group distribution:
  Many-shot (n≥100):    8 classes
  Medium-shot (20≤n):   2 classes
  Few-shot (n<20):      0 classes

✓ 분포 시각화 저장: /content/drive/MyDrive/imbalanced-data-LWCE/image_classification/results/CIFAR10_LT/IR1

In [5]:
# === Cell 3: 모델 및 평가 함수 정의 ===
# 3차 피드백 §4.3 지표: Macro-F1, Balanced Acc, G-Mean, Minority Recall, Worst-class Acc, Per-class Acc
#   → G-Mean/Worst-class/Per-class는 experiment_utils.extended_metrics 로 전 도메인 통일

def _predict(model, loader):
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            yp.extend(model(imgs.to(device)).argmax(dim=1).cpu().tolist())
            yt.extend(labels.tolist())
    return np.array(yt), np.array(yp)


def compute_val_metrics(model, loader, num_classes, class_counts_train=None):
    y_true, y_pred = _predict(model, loader)
    result = {
        'Top1_Acc': float((y_true == y_pred).mean()),
        'F1_Macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
    }
    # Per_Class_Acc / G_Mean / Worst_Acc / Balanced_Acc
    result.update(extended_metrics(y_true, y_pred, num_classes))

    if class_counts_train is not None:
        pca = np.array(result['Per_Class_Acc'])
        sorted_idx = np.argsort(np.array(class_counts_train))[::-1]   # 내림차순 (many→few)
        n = len(sorted_idx)
        result['Many_Acc']   = float(pca[sorted_idx[:n // 3]].mean())
        result['Medium_Acc'] = float(pca[sorted_idx[n // 3:2 * n // 3]].mean())
        # Minority-class Recall = Few 그룹의 recall 평균
        result['Few_Acc']    = float(pca[sorted_idx[2 * n // 3:]].mean())
    return result


def compute_val_acc(model, loader):
    """Optuna 목적함수 / 모델 선택 기준 스칼라 = F1-Macro (balanced_acc 아님)."""
    y_true, y_pred = _predict(model, loader)
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


print('✓ 모델 및 평가 함수 정의 완료')
print('  지표: Top1 / Balanced / F1-Macro / G-Mean / Worst-class / Many·Medium·Few(=Minority Recall)')
print('  Optuna·모델선택 기준: F1-Macro | 그룹: train count tertile split')


✓ 모델 및 평가 함수 정의 완료 (Optuna/checkpoint 기준: F1-Macro, 그룹: tertile split)


In [6]:
# === Cell 4: train_model 함수 정의 ===
# 3차 피드백 §4.7: epoch별 Gradient Norm / Minority·Majority Gradient Ratio 기록 (GradLogger).
#   둘 다 학습이 끝나면 복구 불가능 → 반드시 루프 안에서 잡아야 한다.
#   ResNet+SGD는 grad clipping을 쓰지 않으므로 GradLogger가 total_grad_norm()으로 측정
#   (clip_grad_norm_과 달리 grad를 수정하지 않아 학습 결과에 영향 없음).

def train_model(loss_name: str,
                class_counts: list,
                train_loader,
                val_loader,
                num_classes: int,
                alpha: float = 1.0,
                gamma: float = 2.0,
                eps: float = 0.1,
                tau: float = 1.0,
                epochs: int = 200,
                lr: float = 0.1,
                tag: str = '',
                seed: int = 42):

    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    model = build_resnet32(num_classes).to(device)

    optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=2e-4)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=[160, 180], gamma=0.01)

    criterion = get_clf_loss(loss_name, class_counts, alpha=alpha, gamma=gamma,
                             eps=eps, tau=tau)
    glog = GradLogger(class_counts, num_classes, device)

    best_val_acc = 0.0
    best_model_state = None
    # 주의: history 키 'val_balanced_acc'는 레거시 이름. 실제 값은 F1-Macro.
    history = {'epoch': [], 'train_loss': [], 'val_balanced_acc': [],
               'grad_norm': [], 'grad_many': [], 'grad_few': [], 'grad_ratio': []}

    pbar = tqdm(range(epochs),
                desc=f'{loss_name} (α={alpha:.2f}, γ={gamma:.2f}, ε={eps:.2f}, τ={tau:.2f})',
                leave=False)

    for epoch in pbar:
        model.train()
        glog.reset()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            logits.retain_grad()                 # 샘플별 로짓 기울기 관측 (Proposition 4)
            loss = criterion(logits, labels)
            loss.backward()
            glog.update(logits, labels, model)   # optimizer.step() 전에 호출
            optimizer.step()
            train_loss += loss.item() * labels.size(0)

        train_loss /= len(train_loader.dataset)
        val_f1 = compute_val_acc(model, val_loader)      # F1-Macro

        history['epoch'].append(epoch)
        history['train_loss'].append(train_loss)
        history['val_balanced_acc'].append(val_f1)
        for k, v in glog.epoch_end().items():
            history[k].append(v)

        if val_f1 > best_val_acc:
            best_val_acc = val_f1
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        scheduler.step()
        pbar.update()

    if best_model_state:
        model.load_state_dict(best_model_state)
    model.eval()

    return model, history, best_val_acc


print('✓ train_model 함수 정의 완료 (SGD/MultiStepLR, 선택 기준: F1-Macro)')
print('  기록: train_loss, val F1-Macro, grad_norm, grad_many/few, grad_ratio(=Few/Many)')


✓ train_model 함수 정의 완료


In [7]:
# === Cell 5: Optuna 탐색 (alpha / gamma / eps / tau) ===
# 3차 피드백 §3.3: proxy는 train의 stratified 부분집합, 선택 기준은 val F1-Macro.
#                 test set은 파라미터 선택에 일절 쓰지 않는다.
#            §4.6: study.trials 전체를 저장 → sensitivity 곡선을 추가 학습 없이 확보.
os.environ['TQDM_DISABLE'] = '1'

PROXY_EPOCHS = 20
PROXY_SUBSET_RATIO = 0.20
PROXY_MIN_PER_CLASS = 5
# 탐색 그리드는 experiment_utils.OPTUNA_GRIDS로 전 도메인 통일
#   (1D 30 trials, combined만 2D 10x6=60). n_trials는 grid 크기와 정확히 일치해야 함.

# ── 체크포인트 로드 ──────────────────────────────────
if os.path.exists(CKPT_OPTUNA):
    with open(CKPT_OPTUNA) as f:
        optuna_best = {int(k): v for k, v in json.load(f).items()}
    print(f'Optuna 체크포인트 로드: IR={sorted(optuna_best.keys())} 완료됨')
else:
    optuna_best = {}
    print('Optuna 체크포인트 없음 — 새로 시작')

if os.path.exists(CKPT_OPTUNA_TRIALS):
    with open(CKPT_OPTUNA_TRIALS) as f:
        optuna_trials = {int(k): v for k, v in json.load(f).items()}
else:
    optuna_trials = {}

print(f'\nOptuna 탐색 시작 (proxy: {PROXY_EPOCHS} epochs, subset={PROXY_SUBSET_RATIO})')
print('=' * 60)

for ir in IR_LIST:
    if ir in optuna_best:
        print(f'[IR={ir}] 스킵 (완료됨)')
        continue

    print(f'\n[IR={ir}] Optuna 탐색 중...')

    train_loader, val_loader, _, class_counts = load_cifar_lt_loaders(
        ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    # stratified proxy: 클래스별 최소 PROXY_MIN_PER_CLASS개 보장 → tail 클래스 소멸 방지
    _base_sub    = train_loader.dataset                       # Subset(full, train_indices)
    _all_targets = np.array(_base_sub.dataset.targets)
    _sub_labels  = _all_targets[np.array(_base_sub.indices)]   # proxy 후보 위치별 라벨
    _rng = np.random.RandomState(SEED)
    _proxy_pos = []
    for _c in np.unique(_sub_labels):
        _pos_c  = np.where(_sub_labels == _c)[0]
        _n_take = max(int(round(PROXY_SUBSET_RATIO * len(_pos_c))),
                      min(len(_pos_c), PROXY_MIN_PER_CLASS))
        _n_take = min(_n_take, len(_pos_c))
        _proxy_pos.extend(_rng.choice(_pos_c, size=_n_take, replace=False).tolist())
    _rng.shuffle(_proxy_pos)
    proxy_train_ds = Subset(_base_sub, _proxy_pos)
    proxy_train_loader = DataLoader(proxy_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    print(f'  proxy: {len(_proxy_pos)} samples, {len(np.unique(_sub_labels))} classes '
          f'(min {PROXY_MIN_PER_CLASS}/class 보장)')

    best_ir, trials_ir = {}, {}
    for loss_name, grid in OPTUNA_GRIDS.items():
        n_trials = grid_n_trials(grid)

        def objective(trial, _ln=loss_name, _g=grid):
            params = {p: trial.suggest_float(p, min(vals), max(vals))
                      for p, vals in _g.items()}
            model, _, _ = train_model(_ln, class_counts, proxy_train_loader, val_loader,
                                      NUM_CLASSES, epochs=PROXY_EPOCHS,
                                      tag=f'optuna_{_ln}', **params)
            return compute_val_acc(model, val_loader)

        study = optuna.create_study(direction='maximize',
                                    sampler=GridSampler(grid),   # MedianPruner 병행 금지
                                    study_name=f'{DATASET}_ir{ir}_{loss_name}')
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
        best_ir[loss_name] = dict(study.best_params)
        # 'if t.value is not None' — 0.0 trial이 필터링되는 버그 방지
        trials_ir[loss_name] = [{'params': t.params, 'value': t.value}
                                for t in study.trials if t.value is not None]
        ps = ', '.join(f'{k}={v:.3f}' for k, v in study.best_params.items())
        print(f'  {loss_name:9s} {ps}  F1={study.best_value:.4f}')

    optuna_best[ir]   = best_ir
    optuna_trials[ir] = trials_ir
    with open(CKPT_OPTUNA, 'w') as f:
        json.dump({str(k): v for k, v in optuna_best.items()}, f, indent=2)
    with open(CKPT_OPTUNA_TRIALS, 'w') as f:
        json.dump({str(k): v for k, v in optuna_trials.items()}, f, indent=2)
    print(f'  체크포인트 저장 → {CKPT_OPTUNA}')

os.environ['TQDM_DISABLE'] = '0'
print('\n✓ Optuna 탐색 완료 (ce/wce/sqce/lwce/cb는 파라미터 없음 → 탐색 불필요)')


Optuna 체크포인트 로드: IR=[10, 50, 100] 완료됨
  IR=10: pwce α=0.300, plwce α=1.368, focal γ=1.211
  IR=50: pwce α=0.547, plwce α=3.105, focal γ=2.684
  IR=100: pwce α=0.547, plwce α=2.526, focal γ=2.053

Optuna 탐색 시작 (proxy: 20 epochs, subset=0.2)
[IR=10] 스킵 (완료됨)
[IR=50] 스킵 (완료됨)
[IR=100] 스킵 (완료됨)

✓ Optuna 탐색 완료 — pwce, plwce, focal 최적화됨
  (ce, lwce, cb는 기본값 사용)


In [8]:
# === Cell 6: 전체 Loss × IR × Seed 비교 실험 ===
# 결과는 기존 위치에 버전 접미사 없이 저장. 11종 + gradient/per-class/history 포함이라
# 옛 8종 파일과 스키마가 다르므로, Drive의 기존 결과는 미리 정리(보관)한 뒤 실행할 것.

# ── 체크포인트 로드 ──────────────────────────────────
if os.path.exists(CKPT_RESULTS):
    with open(CKPT_RESULTS) as f:
        ckpt_data = json.load(f)
    print(f'결과 체크포인트 로드: {len(ckpt_data)}개 완료된 실행')
else:
    ckpt_data = {}
    print('체크포인트 없음 — 새로 시작')

# ── 완료된 결과 복원 ────────────────────────────────
all_results   = {ir: {l: {} for l in LOSS_CONFIGS} for ir in IR_LIST}
all_histories = {ir: {l: {} for l in LOSS_CONFIGS} for ir in IR_LIST}

for run_key, run_data in ckpt_data.items():
    # run_key 형식: IR{ir}_{loss_name}_s{seed}
    try:
        ir_str, loss_str, seed_str = run_key.split('_', 2)
        ir   = int(ir_str[2:])
        seed = int(seed_str[1:])
        if ir in all_results and loss_str in all_results[ir]:
            all_results[ir][loss_str][seed]   = run_data['metrics']
            all_histories[ir][loss_str][seed] = run_data['history']
    except Exception:
        pass

total_runs = len(IR_LIST) * len(LOSS_CONFIGS) * len(SEEDS)
print(f'\nFull experiment: {len(IR_LIST)} IRs × {len(LOSS_CONFIGS)} losses × {len(SEEDS)} seeds '
      f'= {total_runs} runs')
print(f'완료: {len(ckpt_data)}/{total_runs}')
print('=' * 60)

# ── 실험 실행 ────────────────────────────────────────
for ir in IR_LIST:
    train_loader, val_loader, test_loader, class_counts = load_cifar_lt_loaders(
        ir, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

    for loss_name in LOSS_CONFIGS:
        # optuna_best[ir][loss_name]은 해당 손실의 파라미터만 담고 있음 → 없으면 기본값.
        # sqce는 파라미터 없는 고정 손실(w ∝ 1/√n) — alpha를 넘기지 않는다.
        p  = optuna_best.get(ir, {}).get(loss_name, {})
        kw = dict(alpha=p.get('alpha', 1.0), gamma=p.get('gamma', 2.0),
                  eps=p.get('eps', 0.1), tau=p.get('tau', 1.0))

        for seed in SEEDS:
            run_key = f'IR{ir}_{loss_name}_s{seed}'
            if run_key in ckpt_data:
                continue

            print(f'\n[IR={ir}] {loss_name} seed={seed} '
                  f'(α={kw["alpha"]:.2f}, γ={kw["gamma"]:.2f}, '
                  f'ε={kw["eps"]:.2f}, τ={kw["tau"]:.2f})...')

            model, history, _ = train_model(
                loss_name, class_counts, train_loader, val_loader,
                NUM_CLASSES, epochs=FINAL_EPOCHS,
                tag=f'ir{ir}_{loss_name}_s{seed}', seed=seed, **kw)

            metrics = compute_val_metrics(model, test_loader, NUM_CLASSES,
                                          class_counts_train=class_counts)
            metrics.update(kw)
            # §4.7 Class Weight Distribution 분석용 — 실제 사용된 정규화 가중치 기록
            w = get_clf_loss(loss_name, class_counts, **kw).get_weights()
            metrics['class_weights'] = None if w is None else w.tolist()

            all_results[ir][loss_name][seed]   = metrics
            all_histories[ir][loss_name][seed] = history

            ckpt_data[run_key] = {'metrics': metrics, 'history': history}
            with open(CKPT_RESULTS, 'w') as f:
                json.dump(ckpt_data, f)

            print(f'  Top1={metrics["Top1_Acc"]:.4f} | F1={metrics["F1_Macro"]:.4f} | '
                  f'Worst={metrics["Worst_Acc"]:.4f} | G-Mean={metrics["G_Mean"]:.4f} | '
                  f'Few={metrics.get("Few_Acc", 0):.4f} | '
                  f'grad_ratio={history["grad_ratio"][-1]:.2f} → 저장됨')

print('\n✓ 전체 훈련 완료')


결과 체크포인트 로드: 105개 완료된 실행
  [IR100_cb_s42] Top1=0.4502 F1=0.4121
  [IR100_cb_s43] Top1=0.4581 F1=0.4159
  [IR100_cb_s44] Top1=0.4909 F1=0.4698
  [IR100_cb_s45] Top1=0.4275 F1=0.3797
  [IR100_cb_s46] Top1=0.4064 F1=0.3535
  [IR100_ce_s42] Top1=0.4345 F1=0.3721
  [IR100_ce_s43] Top1=0.4983 F1=0.4614
  [IR100_ce_s44] Top1=0.4808 F1=0.4443
  [IR100_ce_s45] Top1=0.4881 F1=0.4478
  [IR100_ce_s46] Top1=0.4656 F1=0.4154
  [IR100_focal_s42] Top1=0.5187 F1=0.4904
  [IR100_focal_s43] Top1=0.4640 F1=0.4112
  [IR100_focal_s44] Top1=0.5010 F1=0.4649
  [IR100_focal_s45] Top1=0.4856 F1=0.4456
  [IR100_focal_s46] Top1=0.4571 F1=0.4066
  [IR100_lwce_s42] Top1=0.4851 F1=0.4481
  [IR100_lwce_s43] Top1=0.4857 F1=0.4465
  [IR100_lwce_s44] Top1=0.4864 F1=0.4413
  [IR100_lwce_s45] Top1=0.4749 F1=0.4373
  [IR100_lwce_s46] Top1=0.5130 F1=0.4756
  [IR100_plwce_s42] Top1=0.4855 F1=0.4451
  [IR100_plwce_s43] Top1=0.4773 F1=0.4419
  [IR100_plwce_s44] Top1=0.5045 F1=0.4724
  [IR100_plwce_s45] Top1=0.5154 F1=0.4815
  

wce (α=1.00, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.7227 | F1=0.7249 | Few=0.6825 → 저장됨

[IR=10] wce seed=43...


wce (α=1.00, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.7316 | F1=0.7339 | Few=0.6947 → 저장됨

[IR=10] wce seed=44...


...

  Top1=0.4170 | F1=0.3625 | Few=0.1520 → 저장됨

[IR=100] wce seed=45...


wce (α=1.00, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.4112 | F1=0.3632 | Few=0.1583 → 저장됨

[IR=100] wce seed=46...


wce (α=1.00, γ=2.00):   0%|          | 0/200 [00:00<?, ?it/s]

  Top1=0.4770 | F1=0.4483 | Few=0.2390 → 저장됨

✓ 전체 훈련 완료


In [9]:
# === Cell 7: 결과 집계, 시각화 및 저장 ===

def compute_class_counts_pure(dataset_name, ir):
    """데이터 다운로드 없이 class_counts 계산."""
    K = 10 if dataset_name == 'cifar10' else 100
    n_max = 5000 if dataset_name == 'cifar10' else 500
    rho = ir ** (-1 / (K - 1))
    return [int(n_max * (rho ** i)) for i in range(K)]

def get_group_indices(class_counts):
    """Training count 기준 상위/중위/하위 1/3 인덱스 반환."""
    counts = np.array(class_counts)
    sorted_idx = np.argsort(counts)[::-1]
    n = len(sorted_idx)
    return sorted_idx[:n // 3], sorted_idx[n // 3 : 2 * n // 3], sorted_idx[2 * n // 3:]

print(f'\n최종 결과 요약 (mean ± std, 그룹=상위/중위/하위 1/3 tertile split)')
print('=' * 105)

agg_results = {}
colors_loss = plt.cm.tab10(np.linspace(0, 1, len(LOSS_CONFIGS)))

for ir in IR_LIST:
    agg_results[ir] = {}
    cc_ir = compute_class_counts_pure(DATASET, ir)
    many_idx, medium_idx, few_idx = get_group_indices(cc_ir)

    print(f'\nIR={ir}:')
    print(f'{"Loss":15s} | {"Top1":^16} | {"Balanced":^16} | {"F1-Macro":^16} | {"Many":^8} | {"Medium":^8} | {"Few":^8} | n')
    print('-' * 115)

    for loss_name in LOSS_CONFIGS:
        seed_metrics = [all_results[ir][loss_name][s]
                        for s in SEEDS if s in all_results[ir][loss_name]]
        if not seed_metrics:
            print(f'{loss_name:15s} | (미완료)')
            continue

        # Per_Class_Acc로 그룹별 acc 재계산 (체크포인트 구버전 호환)
        for m in seed_metrics:
            pca = np.array(m.get('Per_Class_Acc', []))
            if len(pca) > 0:
                m['Many_Acc']   = float(pca[many_idx].mean())
                m['Medium_Acc'] = float(pca[medium_idx].mean())
                m['Few_Acc']    = float(pca[few_idx].mean())

        agg = {}
        for key in ['Top1_Acc', 'Balanced_Acc', 'F1_Macro', 'G_Mean', 'Worst_Acc',
                    'Many_Acc', 'Medium_Acc', 'Few_Acc']:
            vals = [m.get(key, 0.0) for m in seed_metrics]
            agg[f'{key}_mean'] = float(np.mean(vals))
            agg[f'{key}_std']  = float(np.std(vals))
        agg_results[ir][loss_name] = agg

        n = len(seed_metrics)
        print(f'{loss_name:15s} | '
              f'{agg["Top1_Acc_mean"]:.4f}±{agg["Top1_Acc_std"]:.4f} | '
              f'{agg["Balanced_Acc_mean"]:.4f}±{agg["Balanced_Acc_std"]:.4f} | '
              f'{agg["F1_Macro_mean"]:.4f}±{agg["F1_Macro_std"]:.4f} | '
              f'{agg["Many_Acc_mean"]:.4f}   | '
              f'{agg["Medium_Acc_mean"]:.4f}   | '
              f'{agg["Few_Acc_mean"]:.4f}   | {n}')

    # ── JSON 저장 ──────────────────────────────────────
    os.makedirs(f'{RESULTS_BASE}/IR{ir}', exist_ok=True)
    with open(f'{RESULTS_BASE}/IR{ir}/results_agg.json', 'w') as f:
        json.dump(agg_results[ir], f, indent=2)
    per_seed_data = {l: {str(s): all_results[ir][l].get(s, {}) for s in SEEDS} for l in LOSS_CONFIGS}
    with open(f'{RESULTS_BASE}/IR{ir}/results_per_seed.json', 'w') as f:
        json.dump(per_seed_data, f, indent=2)

    # ── Excel 저장 ────────────────────────────────────
    summary_rows, per_seed_rows = [], []
    for loss_name in LOSS_CONFIGS:
        if loss_name not in agg_results[ir]:
            continue
        agg = agg_results[ir][loss_name]
        summary_rows.append({
            'Loss':          loss_name,
            'Top1_Mean':     f"{agg['Top1_Acc_mean']:.4f}",
            'Top1_Std':      f"{agg['Top1_Acc_std']:.4f}",
            'Balanced_Mean': f"{agg['Balanced_Acc_mean']:.4f}",
            'Balanced_Std':  f"{agg['Balanced_Acc_std']:.4f}",
            'F1_Macro_Mean': f"{agg['F1_Macro_mean']:.4f}",
            'F1_Macro_Std':  f"{agg['F1_Macro_std']:.4f}",
            'Many_Mean':     f"{agg['Many_Acc_mean']:.4f}",
            'Medium_Mean':   f"{agg['Medium_Acc_mean']:.4f}",
            'Few_Mean':      f"{agg['Few_Acc_mean']:.4f}",
        })
        for seed in SEEDS:
            m = all_results[ir][loss_name].get(seed, {})
            if m:
                per_seed_rows.append({
                    'Loss': loss_name, 'Seed': seed,
                    'Top1_Acc':     f"{m['Top1_Acc']:.4f}",
                    'Balanced_Acc': f"{m['Balanced_Acc']:.4f}",
                    'F1_Macro':     f"{m['F1_Macro']:.4f}",
                    'Many_Acc':     f"{m.get('Many_Acc', 0):.4f}",
                    'Medium_Acc':   f"{m.get('Medium_Acc', 0):.4f}",
                    'Few_Acc':      f"{m.get('Few_Acc', 0):.4f}",
                })

    with pd.ExcelWriter(f'{RESULTS_BASE}/IR{ir}/results.xlsx', engine='openpyxl') as writer:
        pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary_Agg', index=False)
        pd.DataFrame(per_seed_rows).to_excel(writer, sheet_name='Per_Seed', index=False)

# ── 학습 곡선 (mean ± std shading) ─────────────────────
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4))
if len(IR_LIST) == 1:
    axes = [axes]

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    for l_idx, loss_name in enumerate(LOSS_CONFIGS):
        seed_hists = [all_histories[ir][loss_name][s]
                      for s in SEEDS if s in all_histories[ir][loss_name]]
        if not seed_hists:
            continue
        arr  = np.array([h['val_balanced_acc'] for h in seed_hists])
        mean = arr.mean(0)
        std  = arr.std(0)
        ep   = np.arange(len(mean))
        ax.plot(ep, mean, label=loss_name, color=colors_loss[l_idx], alpha=0.85)
        ax.fill_between(ep, mean - std, mean + std, color=colors_loss[l_idx], alpha=0.15)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val F1-Macro')
    ax.set_title(f'IR={ir}')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.suptitle(f'{DATASET.upper()}-LT Training Curves (mean±std, n≤{len(SEEDS)} seeds)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/training_curves_all_irs.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ 학습 곡선 저장')

# ── F1-Macro 비교 막대 그래프 ────────────────────────
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4), sharey=False)
if len(IR_LIST) == 1:
    axes = [axes]

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    losses = [l for l in LOSS_CONFIGS if l in agg_results[ir]]
    means  = [agg_results[ir][l]['F1_Macro_mean'] for l in losses]
    stds   = [agg_results[ir][l]['F1_Macro_std']  for l in losses]
    x = np.arange(len(losses))
    ax.bar(x, means, yerr=stds, capsize=5, alpha=0.78,
           color=[colors_loss[LOSS_CONFIGS.index(l)] for l in losses])
    ce_mean = agg_results[ir].get('ce', {}).get('F1_Macro_mean')
    if ce_mean is not None:
        ax.axhline(ce_mean, color='gray', linestyle='--', linewidth=1,
                   alpha=0.7, label='CE baseline')
        ax.legend(fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(losses, rotation=30, ha='right')
    ax.set_ylabel('F1-Macro')
    ax.set_title(f'IR={ir}')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'{DATASET.upper()}-LT F1-Macro (mean±std, n≤{len(SEEDS)} seeds)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/f1_macro_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ F1-Macro 비교 그래프 저장')

# ── Many/Medium/Few 그룹별 정확도 비교 ──────────────────
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(15, 4))
if len(IR_LIST) == 1:
    axes = [axes]

width = 0.25
group_colors = {'Many': '#2ecc71', 'Medium': '#f39c12', 'Few': '#e74c3c'}

for ax_idx, ir in enumerate(IR_LIST):
    ax = axes[ax_idx]
    losses = [l for l in LOSS_CONFIGS if l in agg_results[ir]]
    x = np.arange(len(losses))
    for g_idx, (group, key) in enumerate([('Many',   'Many_Acc_mean'),
                                           ('Medium', 'Medium_Acc_mean'),
                                           ('Few',    'Few_Acc_mean')]):
        vals = [agg_results[ir][l].get(key, 0.0) for l in losses]
        ax.bar(x + (g_idx - 1) * width, vals, width, label=group,
               color=group_colors[group], alpha=0.78)
    ax.set_xticks(x)
    ax.set_xticklabels(losses, rotation=30, ha='right')
    ax.set_ylabel('Accuracy')
    ax.set_title(f'IR={ir}')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle(f'{DATASET.upper()}-LT Many/Medium/Few Accuracy (tertile split)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{RESULTS_BASE}/group_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('✓ 그룹별 정확도 그래프 저장')

print(f'\n✓ 모든 결과 저장 완료')
print(f'  JSON (집계):   {RESULTS_BASE}/IR*/results_agg.json')
print(f'  JSON (seed별): {RESULTS_BASE}/IR*/results_per_seed.json')
print(f'  Excel:         {RESULTS_BASE}/IR*/results.xlsx  (Summary_Agg / Per_Seed 시트)')
print(f'  Checkpoint:    {CKPT_RESULTS}')
print(f'  PNG: training_curves / f1_macro_comparison / group_accuracy_comparison')


최종 결과 요약 (mean ± std, 그룹=상위/중위/하위 1/3 tertile split)

IR=10:
Loss            |       Top1       |     Balanced     |     F1-Macro     |   Many   |  Medium  |   Few    | n
-------------------------------------------------------------------------------------------------------------------
ce              | 0.7043±0.0057 | 0.7043±0.0057 | 0.7058±0.0057 | 0.8535   | 0.6429   | 0.6384   | 5
wce             | 0.7246±0.0137 | 0.7246±0.0137 | 0.7268±0.0138 | 0.8485   | 0.6497   | 0.6878   | 5
pwce            | 0.7117±0.0099 | 0.7117±0.0099 | 0.7131±0.0099 | 0.8560   | 0.6337   | 0.6620   | 5
sqce            | 0.7185±0.0118 | 0.7185±0.0118 | 0.7201±0.0120 | 0.8533   | 0.6423   | 0.6746   | 5
lwce            | 0.7123±0.0036 | 0.7123±0.0036 | 0.7139±0.0035 | 0.8598   | 0.6411   | 0.6552   | 5
plwce           | 0.7120±0.0079 | 0.7120±0.0079 | 0.7134±0.0078 | 0.8587   | 0.6379   | 0.6575   | 5
cb              | 0.7241±0.0165 | 0.7241±0.0165 | 0.7260±0.0169 | 0.8516   | 0.6450   | 0.6878   | 5
focal

In [ ]:
# === Cell 8: 3차 피드백 §4.5~4.7 분석 (mechanism / sensitivity / ablation) ===
# 통계검정(Friedman/Holm)은 '데이터셋을 블록'으로 두는 것이 정석이므로
# 전 도메인 결과가 모인 뒤 통합 스크립트에서 수행한다 (여기서는 생략).
from itertools import cycle

ANALYSIS_DIR = f'{RESULTS_BASE}/analysis'
os.makedirs(ANALYSIS_DIR, exist_ok=True)

# ── (1) §4.7 Optimization Stability — IR별 Gradient Norm & Few/Many Ratio ──
# "성능 향상만으로는 Weight Explosion 완화를 증명 못 한다"는 지적에 대한 직접 증거.
colors_loss = plt.cm.tab20(np.linspace(0, 1, len(LOSS_CONFIGS)))
fig, axes = plt.subplots(len(IR_LIST), 3, figsize=(18, 4.5 * len(IR_LIST)))
axes = np.atleast_2d(axes)
for r, ir in enumerate(IR_LIST):
    for l_idx, loss_name in enumerate(LOSS_CONFIGS):
        hs = [all_histories[ir][loss_name][s] for s in SEEDS
              if s in all_histories[ir][loss_name]]
        if not hs or 'grad_norm' not in hs[0]:
            continue
        for ax, key in zip(axes[r], ['train_loss', 'grad_norm', 'grad_ratio']):
            arr = np.array([h[key] for h in hs], dtype=float)
            mean, std = np.nanmean(arr, 0), np.nanstd(arr, 0)
            ep = np.arange(len(mean))
            ax.plot(ep, mean, label=loss_name, color=colors_loss[l_idx], alpha=0.85)
            ax.fill_between(ep, mean - std, mean + std, color=colors_loss[l_idx], alpha=0.12)
    axes[r][1].set_yscale('log')      # WCE/PWCE의 explosion을 보려면 로그 스케일 필요
    axes[r][2].set_yscale('log')
    axes[r][2].axhline(1.0, color='k', linestyle=':', linewidth=1, alpha=0.6)
    axes[r][0].set_ylabel(f'IR={ir}', fontsize=11, fontweight='bold')
    for ax, t in zip(axes[r], ['Train Loss', 'Gradient Norm (log)',
                               'Few/Many Gradient Ratio (log)']):
        ax.set_xlabel('Epoch'); ax.set_title(f'{t}  [IR={ir}]', fontsize=10); ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(f'{ANALYSIS_DIR}/optimization_stability.png', dpi=150, bbox_inches='tight')
plt.close()

print('■ Gradient 통계 (마지막 epoch, seed 평균)')
print(f'{"IR":>5} | {"Loss":10s} | {"grad_norm":>10} | {"Few/Many":>9}')
print('-' * 44)
grad_summary = {}
for ir in IR_LIST:
    for loss_name in LOSS_CONFIGS:
        hs = [all_histories[ir][loss_name][s] for s in SEEDS
              if s in all_histories[ir][loss_name]]
        if not hs or 'grad_norm' not in hs[0]:
            continue
        gn = float(np.mean([h['grad_norm'][-1] for h in hs]))
        gr = float(np.nanmean([h['grad_ratio'][-1] for h in hs]))
        grad_summary[f'IR{ir}_{loss_name}'] = {'grad_norm_final': gn, 'grad_ratio_final': gr}
        print(f'{ir:5d} | {loss_name:10s} | {gn:10.4f} | {gr:9.3f}')

# ── (2) §4.7 Class Weight Distribution + Proposition 3 검증 ──
fig, axes = plt.subplots(1, len(IR_LIST), figsize=(6 * len(IR_LIST), 4.5))
axes = np.atleast_1d(axes)
print('\n■ 가중치 동적 범위 w_rare/w_freq — Proposition 3')
print(f'{"IR":>5} | {"Loss":10s} | {"w_rare/w_freq":>14} | 이론')
print('-' * 52)
THEORY = {'wce': 'rho', 'pwce': 'rho^alpha', 'sqce': 'sqrt(rho)',
          'lwce': 'O(log rho)', 'plwce': 'O((log rho)^alpha)',
          'eslwce': '<= LWCE', 'combined': 'O((log rho)^alpha)', 'cb': '-'}
for c, ir in enumerate(IR_LIST):
    _, _, _, class_counts = load_cifar_lt_loaders(ir, batch_size=BATCH_SIZE,
                                                  num_workers=NUM_WORKERS)
    order = np.argsort(np.array(class_counts))[::-1]      # many → few
    markers = cycle(['o', 's', '^', 'v', 'D', 'P', '*', 'X', '<', '>', 'h'])
    ax = axes[c]
    for l_idx, loss_name in enumerate(LOSS_CONFIGS):
        m = next((all_results[ir][loss_name][s] for s in SEEDS
                  if s in all_results[ir][loss_name]), None)
        if not m:
            continue
        w = m.get('class_weights')
        w = np.ones(NUM_CLASSES) if w is None else np.array(w)   # None = 균일(ce/focal/logitadj)
        ax.plot(range(NUM_CLASSES), w[order], marker=next(markers), markersize=2.5,
                label=loss_name, color=colors_loss[l_idx], alpha=0.85, linewidth=1.1)
        if m.get('class_weights') is not None:
            ratio = w[order[-1]] / w[order[0]]
            print(f'{ir:5d} | {loss_name:10s} | {ratio:14.2f} | {THEORY.get(loss_name, "-")}')
    ax.set_yscale('log')
    ax.set_xlabel('Class (sorted: many → few)')
    ax.set_ylabel('Normalized weight (log)')
    ax.set_title(f'Class Weight Distribution [IR={ir}]  '
                 r'$\tilde{w}_c = C w_c / \sum_j w_j$', fontsize=9)
    ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{ANALYSIS_DIR}/class_weight_distribution.png', dpi=150, bbox_inches='tight')
plt.close()

# ── (3) §4.6 Sensitivity (Optuna trial 재활용 — 추가 학습 없음) ──
if os.path.exists(CKPT_OPTUNA_TRIALS):
    with open(CKPT_OPTUNA_TRIALS) as f:
        trials_data = {int(k): v for k, v in json.load(f).items()}
    ONE_D = [('pwce', 'alpha'), ('plwce', 'alpha'), ('eslwce', 'eps'),
             ('focal', 'gamma'), ('logitadj', 'tau')]
    fig, axes = plt.subplots(len(IR_LIST), len(ONE_D),
                             figsize=(4 * len(ONE_D), 3.5 * len(IR_LIST)))
    axes = np.atleast_2d(axes)
    for r, ir in enumerate(IR_LIST):
        for c, (loss_name, param) in enumerate(ONE_D):
            ax = axes[r][c]
            ts = trials_data.get(ir, {}).get(loss_name)
            if not ts:
                ax.set_visible(False); continue
            ts = sorted(ts, key=lambda t: t['params'][param])
            ax.plot([t['params'][param] for t in ts], [t['value'] for t in ts],
                    marker='o', markersize=2.5, color='#2d4860')
            best = max(ts, key=lambda t: t['value'])
            ax.axvline(best['params'][param], color='#e74c3c', linestyle='--',
                       linewidth=1, label=f"best={best['params'][param]:.3f}")
            if param == 'eps':
                ax.set_xscale('log')
            ax.set_xlabel(param); ax.set_ylabel('proxy val F1-Macro')
            ax.set_title(f'{loss_name} [IR={ir}]', fontsize=9)
            ax.legend(fontsize=7); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/sensitivity_1d.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f'\n✓ sensitivity_1d.png (proxy 기준 — 논문 그림엔 full-training sweep 권장)')

    # combined: 2D (alpha x eps) 히트맵
    fig, axes = plt.subplots(1, len(IR_LIST), figsize=(5.5 * len(IR_LIST), 4))
    axes = np.atleast_1d(axes)
    for c, ir in enumerate(IR_LIST):
        ts = trials_data.get(ir, {}).get('combined')
        if not ts:
            axes[c].set_visible(False); continue
        A = sorted({t['params']['alpha'] for t in ts})
        E = sorted({t['params']['eps'] for t in ts})
        Z = np.full((len(E), len(A)), np.nan)
        for t in ts:
            Z[E.index(t['params']['eps']), A.index(t['params']['alpha'])] = t['value']
        im = axes[c].imshow(Z, aspect='auto', origin='lower', cmap='viridis')
        axes[c].set_xticks(range(len(A))); axes[c].set_xticklabels([f'{a:.1f}' for a in A], fontsize=7)
        axes[c].set_yticks(range(len(E))); axes[c].set_yticklabels([f'{e:.2f}' for e in E], fontsize=7)
        axes[c].set_xlabel('alpha (PLWCE)'); axes[c].set_ylabel('eps (ES-LWCE)')
        axes[c].set_title(f'Combined proxy F1 [IR={ir}]', fontsize=9)
        plt.colorbar(im, ax=axes[c])
    plt.tight_layout()
    plt.savefig(f'{ANALYSIS_DIR}/sensitivity_combined_2d.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('✓ sensitivity_combined_2d.png')

# ── (4) §4.5 Ablation 표 ──
print('\n■ Ablation (3차 피드백 §4.5)  [Log compression / Power α / Smoothing ε]')
print(f'{"IR":>5} | {"Method":10s} | {"LC":^3} | {"α":^3} | {"ε":^3} | '
      f'{"F1-Macro":^15} | {"Worst":^6} | {"G-Mean":^6}')
print('-' * 76)
ablation_rows = []
for ir in IR_LIST:
    for name, lc, pw, sm in ABLATION_ROWS:
        ms = [all_results[ir][name][s] for s in SEEDS if s in all_results[ir][name]]
        if not ms:
            continue
        f1 = [m['F1_Macro'] for m in ms]
        wo = float(np.mean([m['Worst_Acc'] for m in ms]))
        gm = float(np.mean([m['G_Mean'] for m in ms]))
        print(f'{ir:5d} | {name:10s} | {lc:^3} | {pw:^3} | {sm:^3} | '
              f'{np.mean(f1):.4f}±{np.std(f1):.4f} | {wo:6.4f} | {gm:6.4f}')
        ablation_rows.append({'IR': ir, 'Method': name, 'LogCompression': lc,
                              'Power_alpha': pw, 'Smoothing_eps': sm,
                              'F1_Macro_mean': float(np.mean(f1)),
                              'F1_Macro_std': float(np.std(f1)),
                              'Worst_Acc': wo, 'G_Mean': gm})

with open(f'{ANALYSIS_DIR}/analysis_summary.json', 'w') as f:
    json.dump({'grad_summary': grad_summary, 'ablation': ablation_rows}, f, indent=2)
print(f'\n✓ 분석 저장 → {ANALYSIS_DIR}/')
print('  PNG: optimization_stability / class_weight_distribution / '
      'sensitivity_1d / sensitivity_combined_2d')
